# 00 — Baseline (the messy "before" picture)

This notebook is deliberately not clean. It is committed as-is so `reports/module-1.md` can point at it and show what the packaged `src/prodml/` code replaced. Restart & Run All should work top to bottom.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No real NYC TLC parquet in this environment -> synthesize something with
# the same shape. Swap this cell for `pd.read_parquet('green_tripdata_2023-01.parquet')`
# once you have the real file (nyc.gov/site/tlc/about/tlc-trip-record-data.page).
rng = np.random.default_rng(42)
n = 20000
pu = rng.integers(1, 30, n)
do = rng.integers(1, 30, n)
dist = np.round(rng.gamma(2.0, 1.8, n), 2)
duration = np.clip(4 + dist*2.6 + np.abs(pu-do)/30*25 + rng.normal(0,3,n), 1, 180)
df = pd.DataFrame({'PULocationID': pu, 'DOLocationID': do, 'trip_distance': dist, 'duration': duration})
df.head()

In [ ]:
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

dv = DictVectorizer()
X_train = dv.fit_transform(train_df[['PU_DO','trip_distance']].to_dict(orient='records'))
X_val = dv.transform(val_df[['PU_DO','trip_distance']].to_dict(orient='records'))
y_train, y_val = train_df['duration'], val_df['duration']

In [ ]:
model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
rmse = mean_squared_error(y_val, preds) ** 0.5
print(f'Validation MAE: {mae:.4f} min')
print(f'Validation RMSE: {rmse:.4f} min')

In [ ]:
import pickle, pathlib
pathlib.Path('../models').mkdir(exist_ok=True)
with open('../models/baseline.pkl', 'wb') as f:
    pickle.dump({'vectorizer': dv, 'model': model}, f)
print('saved models/baseline.pkl')